In [2]:
from pathlib import Path



In [6]:
# path where the data is stored
path_hidrosur = Path('/home/casadoj/Data/Hidrosur')
path_in = path_hidrosur / 'raw' / 'timeseries'

In [20]:
def split_files(file: Path, output_dir: Path):
    
    output_dir.mkdir(parents=True, exist_ok=True)

    current_file = None
    get_id_next_line = False

    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            clean_line = line.strip()
            
            # 1. Skip empty lines or metadata footers
            if not clean_line or clean_line.startswith("(") or clean_line.startswith("Hora") or clean_line.startswith("-"):
                continue

            # 2. Check for the Header line
            if clean_line.startswith("Estacion"):
                if current_file:
                    current_file.close()
                    current_file = None
                get_id_next_line = True
                header_line = line # Save header to write it into the new file
                continue

            # 3. If the previous line was 'Estacion...', this line contains the ID
            if get_id_next_line:
                station_id = clean_line.split()[0] # Grabs the first part of the line
                print(f"Creating file for Station ID: {station_id}")
                
                current_file = open(output_dir / f"{station_id}.txt", 'w', encoding='utf-8')
                current_file.write(header_line) # Write the saved header first
                current_file.write(line)        # Write the current data line
                get_id_next_line = False
                continue

            # 4. Normal data lines
            if current_file:
                current_file.write(line)

    if current_file:
        current_file.close()

## Gauges

In [21]:
kind = 'aforos'

file = path_in / f'{kind.title()}.txt'
output_dir = path_in / kind
output_dir.mkdir(exist_ok=True)

In [22]:
kind = 'embalses'
split_files(
    file=path_in / f'{kind.title()}.txt',
    output_dir = path_in / kind
)